# CELDA 1 — Configuración del entorno

En esta celda importamos todas las librerías necesarias para el proyecto,
configuramos matplotlib para visualizaciones y mostramos los tickers disponibles.

In [ ]:
# =============================================================================
# CELDA 1 — Configuración del entorno
# =============================================================================

# Importaciones principales
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(font_scale=1.2)
warnings.filterwarnings('ignore')

# Añadir directorio src al path
sys.path.append(os.path.join('..', 'src'))

# Importar módulos del proyecto
import data_collection as dc
import preprocessing as pp
import feature_engineering as fe
import event_study as es

# Mostrar tickers disponibles
print("TICKERS DISPONIBLES:")
for nombre, simbolo in dc.TICKERS.items():
    print(f"  {nombre}: {simbolo}")

print("\nConfiguración del entorno completada.")

# CELDA 2 — Parámetros de descarga

Definimos las fechas de inicio y fin para la descarga de datos,
así como la fecha del evento (captura de Maduro).

In [ ]:
# =============================================================================
# CELDA 2 — Parámetros de descarga
# =============================================================================

# Fechas de descarga
START_DATE = "2020-01-01"
END_DATE = (datetime.now() - timedelta(days=1)).strftime('%Y-%m-%d')
EVENT_DATE = "2026-01-03"
EVENT_DATE_HABIL = "2026-01-05"  # primer día hábil post-captura

print(f"Fecha de inicio: {START_DATE}")
print(f"Fecha de fin: {END_DATE}")
print(f"Fecha del evento: {EVENT_DATE}")
print(f"Primer día hábil post-evento: {EVENT_DATE_HABIL}")

# CELDA 3 — Descarga de datos

Descargamos los precios de cierre ajustados desde Yahoo Finance
para todos los activos financieros definidos en el diccionario TICKERS.

In [ ]:
# =============================================================================
# CELDA 3 — Descarga de datos
# =============================================================================

# Descargar datos
df_precios = dc.descargar_datos(dc.TICKERS, START_DATE, END_DATE)

# Mostrar primeras filas
print("\nPrimeras filas del DataFrame de precios:")
print(df_precios.head())

# Guardar datos crudos
dc.guardar_datos(df_precios, "precios_diarios")

# CELDA 4 — Validación de calidad inicial

Realizamos una validación completa de la calidad de los datos descargados,
verificando precios negativos, fechas duplicadas y valores nulos.

In [ ]:
# =============================================================================
# CELDA 4 — Validación de calidad inicial
# =============================================================================

# Validar calidad de datos
metricas_calidad = dc.validar_calidad_datos(df_precios)

# Mostrar resumen de nulos
print("\nResumen de valores nulos por columna:")
for col, nulos in metricas_calidad['nulos'].items():
    porcentaje = metricas_calidad['porcentaje_nulos'][col]
    print(f"  {col}: {nulos} nulos ({porcentaje:.2f}%)")

# CELDA 5 — Visualización de precios históricos normalizados

Graficamos los precios históricos normalizados (indexados a 100) para
diferentes grupos de activos: índices bursátiles, petróleo, acciones y metales.

In [ ]:
# =============================================================================
# CELDA 5 — Visualización de precios históricos normalizados
# =============================================================================

# Normalizar precios a 100
df_normalizado = df_precios / df_precios.iloc[0] * 100

# Definir grupos de activos
grupos = {
    'Índices Bursátiles': ['SP500', 'COLCAP', 'BOVESPA'],
    'Petróleo': ['BRENT', 'WTI'],
    'Acciones': ['EXXON', 'CHEVRON'],
    'Metales': ['GOLD', 'COPPER']
}

# Crear directorio para gráficos si no existe
os.makedirs(os.path.join('..', 'data', 'processed', 'graficos'), exist_ok=True)

# Graficar cada grupo
for nombre_grupo, activos in grupos.items():
    plt.figure(figsize=(12, 6))
    
    for activo in activos:
        if activo in df_normalizado.columns:
            plt.plot(df_normalizado.index, df_normalizado[activo], label=activo, linewidth=2)
    
    # Línea vertical en el evento
    plt.axvline(x=pd.to_datetime(EVENT_DATE), color='red', linestyle='--', 
                linewidth=2, label='Captura Maduro')
    
    # Configurar gráfico
    plt.title(f'Precios Normalizados - {nombre_grupo}', fontsize=14)
    plt.xlabel('Fecha', fontsize=12)
    plt.ylabel('Precio Normalizado (Base=100)', fontsize=12)
    plt.legend(loc='best')
    plt.grid(True, alpha=0.3)
    
    # Guardar gráfico
    ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 
                                f'precios_normalizados_{nombre_grupo.lower().replace(" ", "_")}.png')
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"Gráfico de {nombre_grupo} guardado en: {ruta_guardado}")

# CELDA 6 — Cálculo de retornos logarítmicos

Calculamos los retornos logarítmicos diarios para cada activo financiero
y mostramos estadísticas básicas de los retornos.

In [ ]:
# =============================================================================
# CELDA 6 — Cálculo de retornos logarítmicos
# =============================================================================

# Calcular retornos logarítmicos
df_retornos = dc.calcular_retornos_logaritmicos(df_precios)

# Mostrar primeras filas
print("Primeras filas del DataFrame de retornos:")
print(df_retornos.head())

# Estadísticas básicas
print("\nEstadísticas básicas de retornos:")
print(df_retornos.describe().round(6))

# Guardar retornos
dc.guardar_datos(df_retornos, "retornos_diarios")

# CELDA 7 — Pandas Profiling

Generamos un reporte completo con ydata-profiling sobre el DataFrame de retornos
para obtener un análisis exploratorio detallado de los datos.

In [ ]:
# =============================================================================
# CELDA 7 — Pandas Profiling
# =============================================================================

try:
    from ydata_profiling import ProfileReport
    
    # Generar reporte de profiling
    print("Generando reporte de profiling...")
    profile = ProfileReport(df_retornos, 
                           title="Análisis de Retornos Financieros",
                           explorative=True)
    
    # Guardar reporte
    ruta_reporte = os.path.join('..', 'data', 'processed', 'profiling_retornos.html')
    profile.to_file(ruta_reporte)
    
    print(f"Reporte de profiling guardado en: {ruta_reporte}")
    
    # Mostrar resumen de hallazgos
    print("\nResumen de hallazgos principales:")
    print("- El reporte incluye distribuciones, correlaciones y valores faltantes")
    print("- Se pueden identificar outliers y patrones temporales")
    print("- La matriz de correlación muestra relaciones entre activos")
    
except ImportError:
    print("ydata-profiling no está instalado. Instalando...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ydata-profiling"])
    
    # Reintentar importación
    from ydata_profiling import ProfileReport
    
    # Generar reporte de profiling
    print("Generando reporte de profiling...")
    profile = ProfileReport(df_retornos, 
                           title="Análisis de Retornos Financieros",
                           explorative=True)
    
    # Guardar reporte
    ruta_reporte = os.path.join('..', 'data', 'processed', 'profiling_retornos.html')
    profile.to_file(ruta_reporte)
    
    print(f"Reporte de profiling guardado en: {ruta_reporte}")

# CELDA 8 — Detección y visualización de outliers

Detectamos outliers usando el método IQR y visualizamos los resultados
con boxplots para cada activo financiero.

In [ ]:
# =============================================================================
# CELDA 8 — Detección y visualización de outliers
# =============================================================================

# Detectar outliers
df_con_outliers = pp.detectar_outliers_iqr(df_retornos, umbral=3.0)

# Crear boxplots
plt.figure(figsize=(14, 8))

# Preparar datos para boxplot
datos_boxplot = []
etiquetas = []

for columna in df_retornos.columns:
    datos_boxplot.append(df_retornos[columna].values)
    etiquetas.append(columna)

# Crear boxplot
plt.boxplot(datos_boxplot, labels=etiquetas, patch_artist=True,
            boxprops=dict(facecolor='lightblue', color='blue'),
            medianprops=dict(color='red', linewidth=2))

# Marcar outliers en rojo
for i, columna in enumerate(df_retornos.columns):
    outliers = df_con_outliers.loc[df_con_outliers[f'{columna}_es_outlier'], columna]
    if len(outliers) > 0:
        plt.scatter([i+1]*len(outliers), outliers, color='red', alpha=0.5, s=30)

# Configurar gráfico
plt.title('Distribución de Retornos con Outliers', fontsize=14)
plt.xlabel('Activo', fontsize=12)
plt.ylabel('Retorno Logarítmico', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y', alpha=0.3)

# Guardar gráfico
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'boxplots_outliers.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

print(f"Gráfico de boxplots guardado en: {ruta_guardado}")

# Comentar sobre outliers y eventos conocidos
print("\nAnálisis de outliers:")
print("Los outliers detectados pueden corresponder a:")
print("- Eventos geopolíticos extremos (COVID-19, captura de Maduro)")
print("- Crisis financieras o cambios de política monetaria")
print("- Anomalías en los datos que requieren investigación")

# CELDA 9 — Imputación de nulos

Imputamos valores faltantes usando forward fill, que es apropiado para
series financieras donde el precio del día festivo es igual al último precio disponible.

In [ ]:
# =============================================================================
# CELDA 9 — Imputación de nulos
# =============================================================================

# Imputar nulos
df_retornos_imputado = pp.imputar_nulos_forward_fill(df_retornos)

# Verificar que no queden nulos
nulos_restantes = df_retornos_imputado.isnull().sum().sum()
print(f"\nValores nulos restantes después de imputación: {nulos_restantes}")

# Guardar datos imputados
dc.guardar_datos(df_retornos_imputado, "retornos_imputados")

# CELDA 10 — Estadística descriptiva completa

Calculamos estadísticas descriptivas completas para cada activo y
generamos histogramas con curva normal superpuesta.

In [ ]:
# =============================================================================
# CELDA 10 — Estadística descriptiva completa
# =============================================================================

# Generar estadísticas descriptivas
stats = pp.generar_estadisticas_descriptivas(df_retornos_imputado)

# Crear histogramas con curva normal
n_activos = len(df_retornos_imputado.columns)
n_cols = 3
n_rows = (n_activos + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4*n_rows))
axes = axes.flatten()

for i, columna in enumerate(df_retornos_imputado.columns):
    ax = axes[i]
    
    # Histograma
    ax.hist(df_retornos_imputado[columna], bins=50, density=True, 
            alpha=0.7, color='blue', edgecolor='black')
    
    # Curva normal
    x = np.linspace(df_retornos_imputado[columna].min(), 
                    df_retornos_imputado[columna].max(), 100)
    media = df_retornos_imputado[columna].mean()
    std = df_retornos_imputado[columna].std()
    y = (1/(std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - media) / std)**2)
    ax.plot(x, y, 'r-', linewidth=2, label='Normal')
    
    # Configurar gráfico
    ax.set_title(f'{columna}', fontsize=12)
    ax.set_xlabel('Retorno', fontsize=10)
    ax.set_ylabel('Densidad', fontsize=10)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

# Ocultar ejes vacíos
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()

# Guardar gráfico
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'histogramas_retornos.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

print(f"Histogramas guardados en: {ruta_guardado}")

# Identificar distribuciones con fat tails
print("\nAnálisis de curtosis (fat tails):")
for columna in df_retornos_imputado.columns:
    curtosis = stats.loc['curtosis', columna]
    if curtosis > 3:
        print(f"  {columna}: curtosis = {curtosis:.4f} (colas pesadas)")
    else:
        print(f"  {columna}: curtosis = {curtosis:.4f} (colas normales)")

# CELDA 11 — Análisis de correlaciones para redundancia

Analizamos las correlaciones entre activos para identificar pares
con alta correlación (>0.90) que podrían ser redundantes.

In [ ]:
# =============================================================================
# CELDA 11 — Análisis de correlaciones para redundancia
# =============================================================================

# Calcular matriz de correlación
matriz_corr = df_retornos_imputado.corr()

# Crear heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(matriz_corr, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})

# Configurar gráfico
plt.title('Matriz de Correlación entre Activos', fontsize=14)
plt.tight_layout()

# Guardar gráfico
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'matriz_correlacion.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

print(f"Matriz de correlación guardada en: {ruta_guardado}")

# Identificar pares con alta correlación
print("\nPares con correlación > 0.90:")
pares_alta_corr = []

for i in range(len(matriz_corr.columns)):
    for j in range(i+1, len(matriz_corr.columns)):
        corr = matriz_corr.iloc[i, j]
        if abs(corr) > 0.90:
            par = (matriz_corr.columns[i], matriz_corr.columns[j], corr)
            pares_alta_corr.append(par)
            print(f"  {par[0]} - {par[1]}: {par[2]:.4f}")

# Decisión sobre activos redundantes
print("\nDecisión sobre activos redundantes:")
if pares_alta_corr:
    print("Se identificaron activos altamente correlacionados.")
    print("Para el modelamiento, se recomienda mantener solo uno de cada par.")
    print("Por ejemplo, si BRENT y WTI tienen correlación > 0.90, mantener solo BRENT.")
else:
    print("No se identificaron activos con correlación > 0.90.")
    print("Todos los activos aportan información única al modelo.")

# CELDA 12 — Análisis de correlaciones para irrelevancia

Analizamos la correlación punto-biserial de cada variable predictora
con la variable objetivo para identificar variables irrelevantes.

In [ ]:
# =============================================================================
# CELDA 12 — Análisis de correlaciones para irrelevancia
# =============================================================================

# Primero necesitamos calcular los retornos anormales y la variable objetivo
print("Calculando retornos anormales para análisis de relevancia...")

# Calcular retornos anormales
df_ar = es.calcular_ar_todos_activos(df_retornos_imputado, EVENT_DATE)

# Crear variable objetivo
df_target = es.crear_variable_objetivo(df_ar)

# Para cada activo, calcular correlación punto-biserial con su target
print("\nCorrelación punto-biserial de predictores con target:")

for activo in df_retornos_imputado.columns:
    if activo != 'SP500':  # SP500 es el mercado, no tiene target
        target_col = f'target_{activo}'
        if target_col in df_target.columns:
            # Calcular correlación punto-biserial
            corr, p_valor = stats.pointbiserialr(df_target[target_col], df_retornos_imputado[activo])
            
            # Determinar si es relevante
            relevante = abs(corr) > 0.1 and p_valor < 0.05
            
            print(f"  {activo}: corr={corr:.4f}, p={p_valor:.4f} - {'Relevante' if relevante else 'Irrelevante'}")

# Test Mann-Whitney para variables continuas
print("\nTest Mann-Whitney para variables continuas:")

for activo in df_retornos_imputado.columns:
    if activo != 'SP500':
        target_col = f'target_{activo}'
        if target_col in df_target.columns:
            # Separar por clase
            clase_0 = df_retornos_imputado.loc[df_target[target_col] == 0, activo]
            clase_1 = df_retornos_imputado.loc[df_target[target_col] == 1, activo]
            
            # Aplicar test Mann-Whitney
            stat, p_valor = stats.mannwhitneyu(clase_0, clase_1)
            
            # Determinar si hay diferencia significativa
            significativo = p_valor < 0.05
            
            print(f"  {activo}: stat={stat:.2f}, p={p_valor:.4f} - {'Diferencia significativa' if significativo else 'Sin diferencia significativa'}")

# Decisión sobre variables irrelevantes
print("\nDecisión sobre variables irrelevantes:")
print("Las variables con baja correlación punto-biserial (|corr| < 0.1) y p-valor > 0.05")
print("podrían ser irrelevantes para la predicción y considerarse para eliminación.")

# CELDA 13 — Ingeniería de características

Calculamos las features adicionales para el modelamiento:
- Volatilidad histórica (20 días)
- Momentum (5 días)
- Correlación rodante con Brent (30 días)
- Delta VIX
- Indicador de ventana del evento
- Sector del activo

In [ ]:
# =============================================================================
# CELDA 13 — Ingeniería de características
# =============================================================================

print("Calculando features para modelamiento...")

# 1. Volatilidad histórica (20 días)
df_vol = fe.calcular_volatilidad_historica(df_retornos_imputado, ventana=20)

# 2. Momentum (5 días)
df_mom = fe.calcular_momentum(df_retornos_imputado, ventana=5)

# 3. Correlación rodante con Brent (30 días)
df_corr = fe.calcular_correlacion_rodante_brent(df_retornos_imputado, ventana=30)

# 4. Delta VIX
delta_vix = fe.calcular_delta_vix(df_retornos_imputado)

# 5. Combinar features
df_features = pd.concat([df_vol, df_mom, df_corr, delta_vix], axis=1)

# 6. Crear indicador de ventana del evento
df_features = fe.crear_indicador_ventana(df_features, EVENT_DATE)

# 7. Visualizar cada feature
print("\nGenerando visualizaciones de features...")

# Volatilidad
plt.figure(figsize=(12, 6))
for col in df_vol.columns:
    plt.plot(df_vol.index, df_vol[col], label=col, linewidth=1.5)
plt.axvline(x=pd.to_datetime(EVENT_DATE), color='red', linestyle='--', linewidth=2)
plt.title('Volatilidad Histórica (20 días)', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Volatilidad', fontsize=12)
plt.legend(loc='best', fontsize=8)
plt.grid(True, alpha=0.3)
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'volatilidad_historica.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

# Momentum
plt.figure(figsize=(12, 6))
for col in df_mom.columns:
    plt.plot(df_mom.index, df_mom[col], label=col, linewidth=1.5)
plt.axvline(x=pd.to_datetime(EVENT_DATE), color='red', linestyle='--', linewidth=2)
plt.title('Momentum (5 días)', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Momentum', fontsize=12)
plt.legend(loc='best', fontsize=8)
plt.grid(True, alpha=0.3)
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'momentum_5d.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

# Correlación con Brent
plt.figure(figsize=(12, 6))
for col in df_corr.columns:
    plt.plot(df_corr.index, df_corr[col], label=col, linewidth=1.5)
plt.axvline(x=pd.to_datetime(EVENT_DATE), color='red', linestyle='--', linewidth=2)
plt.title('Correlación Rodante con Brent (30 días)', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Correlación', fontsize=12)
plt.legend(loc='best', fontsize=8)
plt.grid(True, alpha=0.3)
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'correlacion_brent.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

# Delta VIX
plt.figure(figsize=(12, 6))
plt.plot(delta_vix.index, delta_vix.values, linewidth=1.5)
plt.axvline(x=pd.to_datetime(EVENT_DATE), color='red', linestyle='--', linewidth=2)
plt.axhline(y=0, color='black', linestyle='-', alpha=0.3)
plt.title('Variación Diaria del VIX', fontsize=14)
plt.xlabel('Fecha', fontsize=12)
plt.ylabel('Delta VIX', fontsize=12)
plt.grid(True, alpha=0.3)
ruta_guardado = os.path.join('..', 'data', 'processed', 'graficos', 'delta_vix.png')
plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
plt.show()

print("\nFeatures calculadas y visualizaciones generadas.")

# CELDA 14 — Construcción del dataset final

Integramos retornos y features en un único DataFrame para modelamiento,
eliminamos filas con NaN y guardamos el dataset final.

In [ ]:
# =============================================================================
# CELDA 14 — Construcción del dataset final
# =============================================================================

# Construir dataset final
df_modelo = fe.construir_dataset_modelamiento(df_retornos_imputado, df_features)

# Mostrar información del dataset
print("\nInformación del dataset final:")
print(f"Shape: {df_modelo.shape}")
print(f"\nTipos de datos:")
print(df_modelo.dtypes)

# Distribución de sectores
print("\nDistribución de sectores:")
sectores = [fe.calcular_sector(col) for col in df_retornos_imputado.columns]
for sector in set(sectores):
    activos = [col for col in df_retornos_imputado.columns if fe.calcular_sector(col) == sector]
    print(f"  {sector}: {len(activos)} activos ({', '.join(activos)})")

# Primeras filas
print("\nPrimeras filas del dataset:")
print(df_modelo.head())

# Guardar dataset
ruta_guardado = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df_modelo.to_csv(ruta_guardado)
print(f"\nDataset guardado en: {ruta_guardado}")

# Verificar que tenemos suficientes variables y registros
print("\nVerificación de requisitos:")
print(f"Número de variables: {len(df_modelo.columns)} (mínimo requerido: 10)")
print(f"Número de registros: {len(df_modelo)} (mínimo requerido: 400)")

if len(df_modelo.columns) >= 10 and len(df_modelo) >= 400:
    print("✓ Se cumplen los requisitos mínimos de variables y registros.")
else:
    print("⚠ No se cumplen los requisitos mínimos. Revisar datos.")